# 🧹 Notebook 2 — Limpieza y Preprocesamiento de Texto

## ¿Por qué limpiar el texto?

Los modelos de ML/NLP aprenden de patrones en el texto. Si el texto tiene "ruido" (HTML, URLs, caracteres raros, mayúsculas inconsistentes), el modelo:
1. Pierde tiempo procesando información irrelevante
2. Aprende cosas incorrectas (ej: `<html>` aparece igual en ham y phishing)
3. Genera vocabularios más grandes de lo necesario

La limpieza no es una ciencia exacta — cada decisión tiene un trade-off:
- Eliminar URLs vs. reemplazarlas por `[URL]` (preservar la *señal* de que había un link)
- Eliminar stopwords vs. mantenerlas (BERT las necesita para entender contexto)

**Regla general:** Lo que hagamos aquí para modelos clásicos (TF-IDF) puede diferir de lo que hagamos para BERT.

---

## 1. Importar Librerías

- **re**: Expresiones regulares — el cuchillo suizo de limpieza de texto
- **html**: Para convertir entidades HTML como `&amp;` → `&`
- **tqdm**: Barra de progreso para operaciones largas sobre DataFrames

In [7]:
import pandas as pd
import numpy as np
import re
import html
import os
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# tqdm en modo terminal — funciona sin ipywidgets
tqdm.pandas(desc='Procesando')

print('✅ Librerías cargadas')

✅ Librerías cargadas


## 2. Cargar el Dataset

Usamos el mismo dataset consolidado del Notebook 1.

In [2]:
DATA_PATH   = '../data/processed/merged_phishing_full.csv'
OUTPUT_PATH = '../data/processed/phishing_clean.csv'

# Cargamos solo las columnas que necesitamos para ahorrar memoria
# subject + body son el texto que limpiaremos
# label es la etiqueta de clase que siempre necesitamos
# source es útil para diagnóstico si algo sale mal
USECOLS = ['source', 'subject', 'body', 'label']

df = pd.read_csv(DATA_PATH, usecols=USECOLS, low_memory=False)

print(f'Cargados: {len(df):,} registros')
print(f'Columnas: {list(df.columns)}')

# Guardar el conteo original para comparar al final
n_original = len(df)
print(f'\nTotal original: {n_original:,}')

Cargados: 131,339 registros
Columnas: ['source', 'subject', 'body', 'label']

Total original: 131,339


## 3. Manejar Valores Nulos

**Decisión:** Rellenar NaN con cadena vacía `''` en lugar de eliminar filas.

**Por qué:** Un email sin asunto sigue siendo un email válido para clasificar. Muchos emails de spam ni siquiera tienen asunto. Eliminar esas filas introduciría sesgo.

**Excepción:** Si `label` es NaN → esa fila no tiene etiqueta y no sirve para entrenamiento → la eliminamos.

In [4]:
print(f'Nulos ANTES de limpiar:')
print(df.isnull().sum())

# Eliminar filas sin etiqueta — sin label no podemos entrenar
n_antes = len(df)
df = df.dropna(subset=['label'])
n_despues = len(df)
if n_antes != n_despues:
    print(f'\n  Eliminadas {n_antes - n_despues:,} filas sin etiqueta.')

# Asegurarnos de que label sea entero (0 o 1)
# A veces puede estar guardado como float (0.0, 1.0) o string ('0', '1')
df['label'] = df['label'].astype(int)

# Verificar que solo haya valores 0 y 1 en label
labels_unicos = df['label'].unique()
print(f'\nValores únicos en label: {sorted(labels_unicos)}')
assert set(labels_unicos).issubset({0, 1}), '❌ Hay etiquetas inválidas en label!'

# Rellenar NaN en texto con cadena vacía
for col in ['subject', 'body']:
    n_nulos = df[col].isna().sum()
    df[col] = df[col].fillna('')  # NaN → ''
    print(f'  {col}: {n_nulos:,} NaN rellenados con cadena vacía')

print(f'\n✅ Nulos DESPUÉS de limpiar:')
print(df.isnull().sum())

Nulos ANTES de limpiar:
source     0
subject    0
body       0
label      0
dtype: int64

Valores únicos en label: [np.int64(0), np.int64(1)]
  subject: 0 NaN rellenados con cadena vacía
  body: 0 NaN rellenados con cadena vacía

✅ Nulos DESPUÉS de limpiar:
source     0
subject    0
body       0
label      0
dtype: int64


## 4. Funciones de Limpieza

Definimos todas las funciones de limpieza aquí antes de aplicarlas.

### Estrategia de limpieza paso a paso:
1. **Decodificar entidades HTML** (`&amp;` → `&`, `&lt;` → `<`) — muchos emails son HTML
2. **Remover etiquetas HTML** (`<b>texto</b>` → `texto`) — el contenido importa, no los tags
3. **Normalizar URLs** (reemplazar por token `URL`) — la *presencia* de URLs es señal, la URL exacta no
4. **Normalizar emails** (reemplazar por token `EMAIL`) — mismo principio
5. **Normalizar números** (reemplazar por token `NUM`) — evitar overfitting a números específicos
6. **Convertir a minúsculas** — `FREE` y `free` deben ser la misma palabra
7. **Remover caracteres especiales** — dejar solo letras, números, espacios
8. **Normalizar espacios** — eliminar espacios múltiples y saltos de línea

In [5]:
# === PATRONES DE EXPRESIONES REGULARES ===
# Compilamos los patrones UNA SOLA VEZ fuera de la función para eficiencia.
# re.compile() pre-compila el patrón → más rápido cuando se aplica millones de veces.

# Patrón para URLs: https?:// captura http:// y https://
# \S+ captura cualquier carácter no-espacio (el resto de la URL)
RE_URL = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)

# Patrón para emails: cadena@cadena.cadena
RE_EMAIL = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

# Patrón para etiquetas HTML: cualquier cosa entre < y >
# [^>]* captura todo excepto >
RE_HTML_TAG = re.compile(r'<[^>]+>')

# Patrón para números: uno o más dígitos
RE_NUMBERS = re.compile(r'\b\d+\b')

# Patrón para caracteres que no son letras, espacios ni tokens especiales
# Mantenemos [ ] para preservar los tokens [URL], [EMAIL], etc.
RE_SPECIAL = re.compile(r'[^a-zA-Z\s\[\]]')

# Patrón para múltiples espacios/tabs/saltos de línea → un espacio
RE_SPACES = re.compile(r'\s+')


def limpiar_texto(text: str) -> str:
    """
    Aplica el pipeline completo de limpieza a un string de texto.
    
    Retorna el texto limpio como string.
    Cada paso está explicado en el orden de aplicación.
    """
    # Verificar que sea string — puede llegar float/None aunque hayamos llenado NaN
    if not isinstance(text, str):
        return ''
    
    # PASO 1: Decodificar entidades HTML
    # html.unescape convierte &amp; → &, &lt; → <, &gt; → >, &#39; → ', etc.
    text = html.unescape(text)
    
    # PASO 2: Remover etiquetas HTML
    # <b>texto</b> → texto | <img src='...'> → ''
    text = RE_HTML_TAG.sub(' ', text)
    
    # PASO 3: Reemplazar URLs por token especial
    # Usamos 'URL' como token visible — el modelo aprenderá que este token
    # es una señal de phishing cuando aparece frecuentemente
    text = RE_URL.sub(' URL ', text)
    
    # PASO 4: Reemplazar direcciones de email por token
    text = RE_EMAIL.sub(' EMAIL ', text)
    
    # PASO 5: Reemplazar números por token
    # '1000000' y '500' son igualmente sospechosos — lo que importa es que hay un número
    text = RE_NUMBERS.sub(' NUM ', text)
    
    # PASO 6: Convertir a minúsculas
    # DESPUÉS de los pasos anteriores para que los tokens [URL] se conviertan a 'url'
    text = text.lower()
    
    # PASO 7: Remover caracteres especiales no relevantes
    # Mantenemos letras, espacios y brackets para preservar 'url', 'email', 'num'
    text = RE_SPECIAL.sub(' ', text)
    
    # PASO 8: Normalizar espacios múltiples
    text = RE_SPACES.sub(' ', text).strip()
    
    return text


# === PRUEBA RÁPIDA ===
ejemplos = [
    "<b>CLICK HERE</b> to claim your FREE prize!! Visit http://win.scam.com/prize",
    "Hi John, please send me the report by 05/15/2024. Thanks, sarah@company.com",
    "&lt;html&gt;You've won $1,000,000! Contact us now &amp; claim your reward&lt;/html&gt;",
]

print('=== PRUEBA DE LIMPIEZA ===')
for ejemplo in ejemplos:
    limpio = limpiar_texto(ejemplo)
    print(f'ORIGINAL : {ejemplo[:80]}')
    print(f'LIMPIO   : {limpio[:80]}')
    print()

=== PRUEBA DE LIMPIEZA ===
ORIGINAL : <b>CLICK HERE</b> to claim your FREE prize!! Visit http://win.scam.com/prize
LIMPIO   : click here to claim your free prize visit url

ORIGINAL : Hi John, please send me the report by 05/15/2024. Thanks, sarah@company.com
LIMPIO   : hi john please send me the report by num num num thanks email

ORIGINAL : &lt;html&gt;You've won $1,000,000! Contact us now &amp; claim your reward&lt;/ht
LIMPIO   : you ve won num num num contact us now claim your reward



## 5. Aplicar Limpieza al Dataset

Aplicamos la función a las columnas `subject` y `body`. 

> **Nota de rendimiento:** `.progress_apply()` es similar a `.apply()` pero muestra una barra de progreso. Para 131k filas puede tardar 2-5 minutos dependiendo de la longitud del texto.

In [8]:
print('Limpiando columna subject...')
try:
    # Intentar con progress_apply (barra de progreso)
    df['subject_clean'] = df['subject'].progress_apply(limpiar_texto)
except:
    # Si no funciona, usar apply normal (sin barra)
    print('  (Sin barra de progreso)')
    df['subject_clean'] = df['subject'].apply(limpiar_texto)

print('\nLimpiando columna body...')
try:
    df['body_clean'] = df['body'].progress_apply(limpiar_texto)
except:
    print('  (Sin barra de progreso)')
    df['body_clean'] = df['body'].apply(limpiar_texto)

# Combinar subject y body en un solo campo de texto
# Esta es la representación que usarán los modelos
# Separar con un espacio — el asunto es parte del contenido del email
df['texto'] = df['subject_clean'] + ' ' + df['body_clean']

# Limpiar espacios extra que puedan haber quedado de la concatenación
df['texto'] = df['texto'].str.strip().str.replace(r'\s+', ' ', regex=True)

# Eliminar filas donde texto quedó completamente vacío después de limpiar
n_antes = len(df)
df = df[df['texto'].str.len() > 0]
n_vacios = n_antes - len(df)
if n_vacios > 0:
    print(f'\n  Eliminados {n_vacios:,} emails vacíos después de limpieza.')

print(f'\n✅ Limpieza completada. {len(df):,} emails resultantes.')

Limpiando columna subject...


Procesando: 100%|██████████| 131339/131339 [00:00<00:00, 224694.70it/s]



Limpiando columna body...


Procesando: 100%|██████████| 131339/131339 [00:16<00:00, 7986.56it/s]



  Eliminados 1 emails vacíos después de limpieza.

✅ Limpieza completada. 131,338 emails resultantes.


## 6. Verificar Calidad de la Limpieza

Comparamos antes vs. después para confirmar que la limpieza funcionó como esperábamos.

In [9]:
# Mostrar 5 ejemplos comparando texto original vs. limpio
print('=== COMPARACIÓN ANTES / DESPUÉS ===')
samples = df.sample(5, random_state=42)

for i, (_, row) in enumerate(samples.iterrows(), 1):
    label_name = 'HAM' if row['label'] == 0 else 'PHISHING'
    print(f'\n--- Email #{i} ({label_name}) ---')
    print(f'BODY ORIGINAL  : {str(row["body"])[:150]}')
    print(f'TEXTO LIMPIO   : {row["texto"][:150]}')

# Estadísticas de reducción de longitud
df['len_original'] = (df['subject'].str.len() + df['body'].str.len())
df['len_limpio']   = df['texto'].str.len()
reduccion_pct = (1 - df['len_limpio'] / df['len_original'].replace(0, np.nan)).mean() * 100

print(f'\n=== ESTADÍSTICAS DE REDUCCIÓN ===')
print(f'  Longitud promedio original : {df["len_original"].mean():.0f} chars')
print(f'  Longitud promedio limpia   : {df["len_limpio"].mean():.0f} chars')
print(f'  Reducción promedio         : {reduccion_pct:.1f}%')

=== COMPARACIÓN ANTES / DESPUÉS ===

--- Email #1 (HAM) ---
BODY ORIGINAL  : On Sat, Jul 20, 2002 at 12:18:57PM +0100, Kenn Humborg wrote:
> > mine, and i is happy person once again. In case you were wondering, the
> > aim of a
TEXTO LIMPIO   : re [ilug] vanquishing the daemons of shell scripting on sat jul num num at num num pm num kenn humborg wrote mine and i is happy person once again in 

--- Email #2 (PHISHING) ---
BODY ORIGINAL  : 
Read hundreds of genuine testimonials from satisfied users all over the world.  

"Ever since I started on your herbal supplement, Sharon says sex is
TEXTO LIMPIO   : enjoy the sex life you deserve read hundreds of genuine testimonials from satisfied users all over the world ever since i started on your herbal suppl

--- Email #3 (HAM) ---
BODY ORIGINAL  :  
Napster 

Stock Quote Notification

The stock price for Napster (NASD:NAPS) has changed in value by -0.48%
today. 


4/11/2007 9:51:04 AM	 
Last Pri
TEXTO LIMPIO   : stock notification for napster

## 7. Guardar Dataset Limpio

Guardamos el dataset limpio para que los siguientes notebooks no tengan que repetir este proceso.

**¿Por qué no Parquet?** CSV es más compatible y fácil de inspeccionar. Para datasets >1GB usaríamos Parquet por rendimiento.

In [10]:
# Seleccionar solo las columnas necesarias para los modelos
# No guardamos columnas intermedias de diagnóstico
cols_guardar = ['source', 'subject', 'body', 'texto', 'label']
df_save = df[cols_guardar].copy()

# Guardar como CSV
df_save.to_csv(OUTPUT_PATH, index=False)

# Verificar que se guardó correctamente
size_mb = os.path.getsize(OUTPUT_PATH) / 1024**2
print(f'✅ Dataset limpio guardado:')
print(f'   Ruta     : {OUTPUT_PATH}')
print(f'   Registros: {len(df_save):,}')
print(f'   Tamaño   : {size_mb:.1f} MB')
print(f'   Columnas : {list(df_save.columns)}')

# Resumen final
print(f'\n=== RESUMEN FINAL ===')
print(f'  Emails originales  : {n_original:,}')
print(f'  Emails limpios     : {len(df_save):,}')
print(f'  Emails eliminados  : {n_original - len(df_save):,}')
print(f'  HAM en dataset limpio    : {(df_save["label"]==0).sum():,}')
print(f'  PHISHING en dataset limpio: {(df_save["label"]==1).sum():,}')

✅ Dataset limpio guardado:
   Ruta     : ../data/processed/phishing_clean.csv
   Registros: 131,338
   Tamaño   : 390.8 MB
   Columnas : ['source', 'subject', 'body', 'texto', 'label']

=== RESUMEN FINAL ===
  Emails originales  : 131,339
  Emails limpios     : 131,338
  Emails eliminados  : 1
  HAM en dataset limpio    : 63,949
  PHISHING en dataset limpio: 67,389


## 8. ¿Qué limpieza usar para BERT?

IMPORTANTE: Para BERT (Notebook 4) usaremos una limpieza **más suave**:

| Paso | Modelos Clásicos | BERT |
|---|---|---|
| Remover HTML | ✅ Sí | ✅ Sí |
| Normalizar URLs | `URL` token | `URL` token |
| Minúsculas | ✅ Sí | ❌ No (BERT tiene su propio tokenizador case-sensitive) |
| Remover stopwords | ✅ Sí | ❌ NO — BERT necesita stopwords para entender contexto |
| Remover puntuación | ✅ Sí | ❌ Parcialmente — BERT usa puntuación |

**Por eso BERT tiene su propio notebook separado con su propio preprocessing.**

---
*Siguiente paso → [03_modelos_clasicos.ipynb](03_modelos_clasicos.ipynb)*